In [6]:
import rasterio
from rasterio.enums import Resampling
import os
import glob
import fiona
import numpy as np
from rasterio.mask import mask
import re

In [25]:
dir = "C:/Users/RDCRLSMC/Desktop/SIRO/Task1/SNODAS/Snowdepth/"  
raster_files = glob.glob(os.path.join(dir,"resampled", '*.tif'))

In [16]:
MCS = os.path.join(dir, "MCS_outline/basin_outline.shp")

with fiona.open(MCS, "r") as shapefile:
    shapes = [feature["geometry"] for feature in shapefile]

In [26]:
for raster in raster_files:
    with rasterio.open(raster) as src:
        print(src.crs)
        print(src.meta)

EPSG:32611
{'driver': 'GTiff', 'dtype': 'int16', 'nodata': None, 'width': 363, 'height': 464, 'count': 1, 'crs': CRS.from_epsg(32611), 'transform': Affine(100.0, 0.0, 572490.0,
       0.0, -100.0, 4877940.0)}
EPSG:32611
{'driver': 'GTiff', 'dtype': 'int16', 'nodata': None, 'width': 363, 'height': 464, 'count': 1, 'crs': CRS.from_epsg(32611), 'transform': Affine(100.0, 0.0, 572490.0,
       0.0, -100.0, 4877940.0)}
EPSG:32611
{'driver': 'GTiff', 'dtype': 'int16', 'nodata': None, 'width': 363, 'height': 464, 'count': 1, 'crs': CRS.from_epsg(32611), 'transform': Affine(100.0, 0.0, 572490.0,
       0.0, -100.0, 4877940.0)}
EPSG:32611
{'driver': 'GTiff', 'dtype': 'int16', 'nodata': None, 'width': 363, 'height': 464, 'count': 1, 'crs': CRS.from_epsg(32611), 'transform': Affine(100.0, 0.0, 572490.0,
       0.0, -100.0, 4877940.0)}
EPSG:32611
{'driver': 'GTiff', 'dtype': 'int16', 'nodata': None, 'width': 363, 'height': 464, 'count': 1, 'crs': CRS.from_epsg(32611), 'transform': Affine(100.0, 0.

In [27]:
out_dir = os.path.join(dir, "outputs")   

# Loop by model# 
for raster in raster_files:
    with rasterio.open(raster) as src:
        out_image, out_transform = rasterio.mask.mask(src, shapes, crop=True, nodata= -9999)
        profile = src.profile
        profile.update(dtype=rasterio.float32,nodata=-9999)
        

        # Include model in output filename
        date = re.search(r"\d{8}", raster).group()
        out_name = f"SNODAS_{date}_basin_clip.tif"
        out_path =   os.path.join(out_dir, out_name)

        with rasterio.open(out_path, "w", **profile) as dest:
            dest.write(out_image)



In [28]:
MCS = os.path.join(dir, "MCS_outline/MCS_outline.shp")

with fiona.open(MCS, "r") as shapefile:
    shapes = [feature["geometry"] for feature in shapefile]

In [29]:
rasters = glob.glob(os.path.join(out_dir, '*.tif'))

for raster in rasters:
        with rasterio.open(raster) as src:
            out_image, out_transform = rasterio.mask.mask(src, shapes, crop=True)
            out_meta = src.meta.copy()

        out_meta.update({
            "driver": "GTiff",
            "height": out_image.shape[1],
            "width": out_image.shape[2],
            "transform": out_transform,
        })

        # Include model in output filename
        out_name = os.path.basename(raster).replace("basin_clip.tif", "MCS.tif")
        out_path = os.path.join(out_dir, out_name)

        with rasterio.open(out_path, "w", **out_meta) as dest:
            dest.write(out_image)